In [3]:
!pip install transformers
!pip install sentencepiece
!pip install datasets


  Using cached fsspec-2024.9.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2024.9.0-py3-none-any.whl (179 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
# Install the specific version of fsspec required by gcsfs
!pip install fsspec==2024.10.0

  Using cached fsspec-2024.10.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2024.10.0-py3-none-any.whl (179 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.9.0
    Uninstalling fsspec-2024.9.0:
      Successfully uninstalled fsspec-2024.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.1.0 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspec 2024.10.0 which is incompatible.


In [5]:
# Install the specific version of fsspec required by gcsfs
!pip install fsspec==2024.10.0


In [6]:
# Update datasets to the latest version
!pip install -U datasets


  Using cached fsspec-2024.9.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2024.9.0-py3-none-any.whl (179 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!pip install evaluate
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import evaluate


# Define dataset paths
train_src = '/content/drive/MyDrive/Code_transfer/train.src'
train_tgt = '/content/drive/MyDrive/Code_transfer/train.tgt'
valid_src = '/content/drive/MyDrive/Code_transfer/valid.src'
valid_tgt = '/content/drive/MyDrive/Code_transfer/valid.tgt'
test_src = '/content/drive/MyDrive/Code_transfer/test.src'
test_tgt = '/content/drive/MyDrive/Code_transfer/test.tgt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.3 MB/s eta 0:00:00


In [9]:
def load_code_translation_data(src_path, tgt_path):
    with open(src_path, 'r') as src_file, open(tgt_path, 'r') as tgt_file:
        src_lines = src_file.readlines()
        tgt_lines = tgt_file.readlines()
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})

# Load datasets
train_dataset = load_code_translation_data(train_src, train_tgt)
valid_dataset = load_code_translation_data(valid_src, valid_tgt)
test_dataset = load_code_translation_data(test_src, test_tgt)


In [10]:
# Load CodeT5 model
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-small")

# Initialize a WordPiece tokenizer for CodeT5 (pre-trained or fine-tuned)
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small", use_fast=True)

# Define encoding function for WordPiece
def encode_wordpiece(example):
    src_enc = tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    tgt_enc = tokenizer(example['tgt'], padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask'], 'labels': tgt_enc['input_ids']}

# Map encoding to datasets
train_wordpiece = train_dataset.map(encode_wordpiece)
valid_wordpiece = valid_dataset.map(encode_wordpiece)
test_wordpiece = test_dataset.map(encode_wordpiece)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

... (2 outputs left out)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# Train the Model

In [11]:
training_args = TrainingArguments(
    output_dir="./results_wordpiece",
    evaluation_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=10_000,
    save_total_limit=2,
    report_to="none"
)

trainer_wordpiece = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_wordpiece,
    eval_dataset=valid_wordpiece
)

trainer_wordpiece.train()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.325400,0.241555
2,0.226300,0.208001
3,0.204500,0.197392


TrainOutput(global_step=825, training_loss=0.2852241440975305, metrics={'train_runtime': 333.9894, 'train_samples_per_second': 9.881, 'train_steps_per_second': 2.47, 'total_flos': 446627944857600.0, 'train_loss': 0.2852241440975305, 'epoch': 3.0})

# Evaluate with BLEU Score



In [12]:
# Install the required jiwer library for WER computation
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.9 MB/s eta 0:00:00


In [13]:
import torch
from evaluate import load

# Define BLEU metric
bleu_metric = load("bleu")
wer_metric = load("wer")

# Check for device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to the same device

# Define function to generate predictions and references
def generate_predictions_and_references(test_data, encoding_function, max_new_tokens=128):
    predictions = []
    references = []
    for example in test_data:
        encoded_example = encoding_function(example)
        input_ids = torch.tensor(encoded_example['input_ids']).unsqueeze(0).to(device)  # Move input_ids to the same device

        # Generate outputs and decode predictions
        outputs = model.generate(input_ids, max_new_tokens=max_new_tokens)
        decoded_pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        decoded_ref = example['tgt']

        predictions.append(decoded_pred)
        references.append(decoded_ref)
    return predictions, references

# Generate predictions and references
predictions_wordpiece, references_wordpiece = generate_predictions_and_references(test_wordpiece, encode_wordpiece)

# Compute BLEU score
bleu_score = bleu_metric.compute(predictions=predictions_wordpiece, references=[[ref] for ref in references_wordpiece])
print(f"BLEU Score for WordPiece encoding: {bleu_score['bleu']}")

# Compute WER score
wer_score = wer_metric.compute(predictions=predictions_wordpiece, references=references_wordpiece)
print(f"WER Score for WordPiece encoding: {wer_score}")

BLEU Score for WordPiece encoding: 0.485500561359427
WER Score for WordPiece encoding: 0.4927536231884058


In [14]:
import torch
from evaluate import load

# Load BLEU and WER metrics
bleu_metric = load("bleu")
wer_metric = load("wer")

# Check for device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to the appropriate device

# Define function to generate predictions and references
def generate_predictions_and_references(test_data, encoding_function, max_new_tokens=128):
    predictions = []
    references = []
    for example in test_data:
        # Encode the example and move input_ids to the device
        encoded_example = encoding_function(example)
        input_ids = torch.tensor(encoded_example['input_ids']).unsqueeze(0).to(device)

        # Generate output and decode the prediction
        outputs = model.generate(input_ids, max_new_tokens=max_new_tokens)
        decoded_pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        decoded_ref = example['tgt']  # Reference (ground truth)

        predictions.append(decoded_pred)
        references.append(decoded_ref)
    return predictions, references

# Generate predictions and references for WordPiece encoding
predictions_wordpiece, references_wordpiece = generate_predictions_and_references(test_wordpiece, encode_wordpiece)

# Display five pairs of predictions and references
print("Sample Predictions and References:")
for i in range(5):
    print(f"\nExample {i+1}:")
    print(f"Prediction: {predictions_wordpiece[i]}")
    print(f"Reference: {references_wordpiece[i]}")

# Compute BLEU score
bleu_score = bleu_metric.compute(predictions=predictions_wordpiece, references=[[ref] for ref in references_wordpiece])
print(f"\nBLEU Score for WordPiece encoding: {bleu_score['bleu']}")

# Compute WER score
wer_score = wer_metric.compute(predictions=predictions_wordpiece, references=references_wordpiece)
print(f"WER Score for WordPiece encoding: {wer_score}")


Sample Predictions and References:

Example 1:
Prediction: #include <stdio.h> @  @ int entry_point ( float ( *speed )  ,  float ( *time )  ,  float ( *temptemp1 )  )   { @ float distance ; @ ; @ distance =  ( *speed )  *  ( *time )  ; @ { @ ( *temptemp1 )  = distance ; @ ; @ } @ return 0 ; @ }

Reference: #include <stdio.h> @  @ int entry_point ( int ( *speed )  ,  int ( *time )  ,  int ( *temptemp1 )  )   { @ int distance ; @ ; @ ; @ distance =  ( *speed )  *  ( *time )  ; @ { @ ( *temptemp1 )  = distance ; @ ; @ } @ return 0 ; @ }


Example 2:
Prediction: #include <stdio.h> @  @ int entry_point ( int ( *a )  ,  int ( *b )  ,  int ( *temptemp1 )  ,  int ( *temptemp2 )  )   { @ int i ; @ int j ; @ int i ; @ ; @ for  ( i = 0 ;  i <  ( *a )  ;  i++ )   { @ { @ ( *temptemp1 )  = i ; @ ; @ } @ } @ for  ( i = 0 ;  i <  ( *
Reference: #include <stdio.h> @  @ int entry_point ( int ( *a )  ,  int ( *b )  ,  int ( *temptemp1 )  ,  int ( *temptemp2 )  )   { @ ; @ ; @ ( *a )  =  ( *a )  +  ( *b )

In [16]:
# Display 5 samples of source, reference, and prediction
print("\nSample Predictions:\n")
for i in range(5):  # Displaying only the first 5 examples
    source = test_wordpiece[i]            # Access source code directly from the list
    reference = references_wordpiece[i]   # Reference C code
    prediction = predictions_wordpiece[i] # Predicted C code

    print(f"Sample {i+1}:")
    print("Source Code (Verilog):\n", source)
    print("Reference Code (C):\n", reference)
    print("Predicted Code (C):\n", prediction)
    print("="*50)



Sample Predictions:

Sample 1:
Source Code (Verilog):
 {'src': "module entry_point ( , ap_start, , ap_done, , ap_idle, , ap_ready, , speed, , time_r, , temptemp1, , temptemp1_ap_vld, , ap_return , ); , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] speed; , input [31:0] time_r; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] ap_return; , reg temptemp1_ap_vld; , wire [31:0] distance_fu_45_p2; , wire ap_ce_reg; , entry_point_mul_32s_32s_32_1_1 #( , .ID( 1 ), , .NUM_STAGE( 1 ), , .din0_WIDTH( 32 ), , .din1_WIDTH( 32 ), , .dout_WIDTH( 32 )) , mul_32s_32s_32_1_1_U1( , .din0(time_r), , .din1(speed), , .dout(distance_fu_45_p2) , ); , always @ (*) begin , if ((ap_start == 1'b1)) begin , temptemp1_ap_vld = 1'b1; , end else begin , temptemp1_ap_vld = 1'b0; , end , end , assign ap_done = ap_start; , assign ap_idle = 1'b1; , assign ap_ready = ap_start; , assign ap_return = 32'd0; , assign temptemp1 = distance_fu_45_p2; , endmodule , mod

# positional encoding

In [20]:
def load_code_translation_data(src_path, tgt_path):
    with open(src_path, 'r') as src_file, open(tgt_path, 'r') as tgt_file:
        src_lines = src_file.readlines()
        tgt_lines = tgt_file.readlines()
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})

# Load datasets
train_data = load_code_translation_data(train_src, train_tgt)
valid_data = load_code_translation_data(valid_src, valid_tgt)
test_data = load_code_translation_data(test_src, test_tgt)


In [21]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

def encode_with_positional_encoding(example):
    src_tokens = tokenizer(example['src'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    tgt_tokens = tokenizer(example['tgt'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")

    # Add positional encoding
    src_tokens['position_ids'] = torch.arange(128).unsqueeze(0)
    tgt_tokens['position_ids'] = torch.arange(128).unsqueeze(0)

    return {
        'input_ids': src_tokens['input_ids'].squeeze(),
        'attention_mask': src_tokens['attention_mask'].squeeze(),
        'labels': tgt_tokens['input_ids'].squeeze()
    }

# Apply encoding to datasets
train_data = train_data.map(encode_with_positional_encoding)
valid_data = valid_data.map(encode_with_positional_encoding)
test_data = test_data.map(encode_with_positional_encoding)


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# CodeT5 Model and Trainer

In [24]:
# Load CodeT5 model
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-small")

# Set training parameters
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=10000,
    save_total_limit=2,
    report_to="none"
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.758000,0.557876
2,0.553400,0.484268
3,0.487700,0.462944


TrainOutput(global_step=825, training_loss=0.7040302473126036, metrics={'train_runtime': 92.7929, 'train_samples_per_second': 35.563, 'train_steps_per_second': 8.891, 'total_flos': 111656986214400.0, 'train_loss': 0.7040302473126036, 'epoch': 3.0})

In [26]:
# Load BLEU and WER metrics
bleu_metric = load("bleu")
wer_metric = load("wer")

def generate_predictions_and_references(test_data, max_new_tokens=128):
    predictions = []
    references = []

    for example in test_data:
        # Convert input_ids to tensor
        input_ids = torch.tensor(example['input_ids']).unsqueeze(0).to(model.device)  # Add batch dimension and move to the model's device

        # Generate model output
        outputs = model.generate(input_ids, max_new_tokens=max_new_tokens)

        # Decode predictions and references
        decoded_pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        decoded_ref = example['tgt']

        predictions.append(decoded_pred)
        references.append(decoded_ref)

    return predictions, references

# Generate predictions and references
predictions, references = generate_predictions_and_references(test_data)

# Compute BLEU score
bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU Score for Positional Encoding: {bleu_score['bleu']}")

# Compute WER score
wer_score = wer_metric.compute(predictions=predictions, references=references)
print(f"WER Score for Positional Encoding: {wer_score}")

# Display sample predictions
print("\nSample Predictions:\n")
for i in range(5):  # Display first 5 samples
    print(f"Source (Verilog): {test_data[i]['src']}")
    print(f"Reference (C): {references[i]}")
    print(f"Prediction: {predictions[i]}")
    print()


BLEU Score for Positional Encoding: 0.5064063908999878
WER Score for Positional Encoding: 0.4803312629399586

Sample Predictions:

Source (Verilog): module entry_point ( , ap_start, , ap_done, , ap_idle, , ap_ready, , speed, , time_r, , temptemp1, , temptemp1_ap_vld, , ap_return , ); , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] speed; , input [31:0] time_r; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] ap_return; , reg temptemp1_ap_vld; , wire [31:0] distance_fu_45_p2; , wire ap_ce_reg; , entry_point_mul_32s_32s_32_1_1 #( , .ID( 1 ), , .NUM_STAGE( 1 ), , .din0_WIDTH( 32 ), , .din1_WIDTH( 32 ), , .dout_WIDTH( 32 )) , mul_32s_32s_32_1_1_U1( , .din0(time_r), , .din1(speed), , .dout(distance_fu_45_p2) , ); , always @ (*) begin , if ((ap_start == 1'b1)) begin , temptemp1_ap_vld = 1'b1; , end else begin , temptemp1_ap_vld = 1'b0; , end , end , assign ap_done = ap_start; , assign ap_idle = 1'b1; , assign ap_ready = ap_start; ,

# BPE

In [28]:
import os
from tokenizers import ByteLevelBPETokenizer
from transformers import RobertaTokenizer

# Define the directory to save the BPE model
save_dir = './bpe_model'

# Create the directory if it does not exist
os.makedirs(save_dir, exist_ok=True)

# Initialize the BPE tokenizer
tokenizer = ByteLevelBPETokenizer()

# Train the BPE tokenizer on your training dataset
tokenizer.train([train_src, train_tgt], vocab_size=30000, min_frequency=2, special_tokens=[
    "<s>", "</s>", "<pad>", "<unk>", "<mask>"
])

# Save the trained tokenizer
tokenizer.save_model(save_dir)

# Load the tokenizer using transformers
tokenizer = RobertaTokenizer.from_pretrained(save_dir)


# Load CodeT5 Model and Prepare for Fine-Tuning

In [ ]:
from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments

# Load the pre-trained CodeT5 model
model = T5ForConditionalGeneration.from_pretrained("Salesforce/codet5-base")

# Set model config for encoder and decoder to use the BPE tokenizer
model.resize_token_embeddings(len(tokenizer))


# the Datasets for Training

In [29]:
from datasets import load_dataset

# Load the datasets (assuming your datasets are in the format of source and target)
train_dataset = load_dataset('text', data_files={'train': train_src})
train_dataset['train'] = train_dataset['train'].map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length'), batched=True)

valid_dataset = load_dataset('text', data_files={'validation': valid_src})
valid_dataset['validation'] = valid_dataset['validation'].map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length'), batched=True)

# Map target C dataset similarly
train_tgt_dataset = load_dataset('text', data_files={'train': train_tgt})
valid_tgt_dataset = load_dataset('text', data_files={'validation': valid_tgt})

# Ensure that the dataset is tokenized properly for the target side (C code)
train_dataset['train'] = train_dataset['train'].map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length'), batched=True)
valid_dataset['validation'] = valid_dataset['validation'].map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length'), batched=True)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

# Train the Model

In [44]:
from datasets import Dataset, DatasetDict

# Load your training and validation data
train_data = {'src': open(train_src).readlines(), 'tgt': open(train_tgt).readlines()}
valid_data = {'src': open(valid_src).readlines(), 'tgt': open(valid_tgt).readlines()}

# Create dataset objects for both
train_dataset = Dataset.from_dict(train_data)
valid_dataset = Dataset.from_dict(valid_data)

# Tokenize the training and validation datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

# Combine them into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': valid_dataset
})

# Now, you can access dataset splits like this:
# dataset['train'] for training
# dataset['validation'] for validation


NameError: name 'tokenize_function' is not defined